# ERCOT selection analysis — T0 (prompt 32; zero new simulations)

Everything here is computed from committed extracts + vendored inputs. Standing caveat rider:
**TL=120 time-truncated MILP incumbents; deltas only with ×-ruler; no cross-system cost-level
comparisons vs RTS.** Rulers are IDENTICAL-INPUT rerun spreads conditional on the CRC execution
environment (mechanism differs from RTS's perturbed-input floors — never conflate).

In [1]:
import gzip, json, os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.labelweight": "bold", "axes.titleweight": "bold",
                     "font.weight": "bold", "text.parse_math": False})
from scipy import stats

ERCOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA = os.path.join(ERCOT, "data", "ercot123_2019")
FOOTER = "ERCOT: DA≡RT; RUC TimeLimit 120 s (12.1% base days truncated); TL=120 incumbent caveat"

eff = pd.read_csv(os.path.join(ERCOT, "analysis", "screening_site_effects.csv"))
eff["site"] = eff["site"].astype(str)
assert len(eff) == 16
v1 = json.load(open(os.path.join(ERCOT, "analysis", "screening_noise_ruler.json")))
print("v1 rulers:", {k: round(v, 1) for k, v in v1.items()})

v1 rulers: {'true_curt': 5479.1, 'h2': 2914.1, 'var_cost': 285620.6, 'reserve': 554.2, 'onoffs': 258.0, 'total_cost': 79451497.0, 'fixed_cost': 79504038.2}


## 1. Ruler bookkeeping — v2 list schema (adjudication RECORDED, not re-opened)

Adjudicated 2026-10-07: SCREENING_ERCOT.md's 7,921 MWh was the PEM-contaminated `overall.csv`
275-pair reading — exactly clean 5,007 + PEM-twin |Δ| 2,914 (= the h2 ruler). Clean true_curt
reading list = **{5,479 (base pair), 5,007 (275 pair)}**; the 397× site-275 headline stands
(2.17 TWh / 5,479). v1 stays frozen as provenance; v2 stores per-pair reading LISTS, never a
bare scalar.

In [2]:
def overall(p): return pd.read_csv(p).iloc[0]
def gens(p):
    g = pd.read_csv(p); g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
    g["Generator"] = g["Generator"].astype(str); return g
def true_curt(p):
    g = gens(p); return float(g[(~g.is_pem) & g.unit_type.isin(["WIND", "PV"])].curtailment_mwh.sum())
def h2_of(p):
    g = gens(p); return float(g[g.is_pem].curtailment_mwh.sum())

E = lambda w, i=None: os.path.join(ERCOT, "extracts", w if i is None else f"{w}/run_index_{i}")
pairs = {  # (label, run A, run B)
    "base_pair": (E("base_2019"), E("screening_ercot", 11)),
    "wind275_pair": (E("screening_ercot", 1), E("screening_ercot", 12)),
}
OBJ = ["true_curt", "h2", "var_cost", "reserve", "onoffs", "total_cost", "fixed_cost"]
def metrics(d):
    o = overall(os.path.join(d, "overall.csv"))
    return {"true_curt": true_curt(os.path.join(d, "gen_summary.csv")),
            "h2": h2_of(os.path.join(d, "gen_summary.csv")),
            "var_cost": float(o["Total generation costs"]),
            "reserve": float(o["Total reserve shortfall"]),
            "onoffs": float(o["Total on/offs"]),
            "total_cost": float(o["Total costs"]),
            "fixed_cost": float(o["Total fixed costs"])}

readings = {k: {} for k in OBJ}
for lab, (a, b) in pairs.items():
    ma, mb = metrics(a), metrics(b)
    for k in OBJ:
        readings[k][lab] = abs(ma[k] - mb[k])
# the adjudication decomposition, re-verified from the committed extracts
contam = abs(overall(os.path.join(E("screening_ercot", 12), "overall.csv"))["Total renewables curtailment"]
             - overall(os.path.join(E("screening_ercot", 1), "overall.csv"))["Total renewables curtailment"])
assert abs(contam - (readings["true_curt"]["wind275_pair"] + readings["h2"]["wind275_pair"])) < 1.0
print(f"decomposition verified: {contam:,.1f} = {readings['true_curt']['wind275_pair']:,.1f} clean "
      f"+ {readings['h2']['wind275_pair']:,.1f} h2")

v2 = {"schema": "v2-list (per-pair reading lists; never a bare scalar)",
      "provenance": "T0 2026-10-08 from the two screening replicate pairs; "
                    "T1 adds design-group keys (base/wind-pocket/pv/nuclear)",
      "replicate_definition": "identical inputs rerun under TL=120 (machine-load-"
                              "dependent incumbents); conditional on CRC env",
      "adjudication_2026_10_07": {
          "contaminated_overall_275pair": float(contam),
          "clean_275pair": readings["true_curt"]["wind275_pair"],
          "pem_twin_h2_275pair": readings["h2"]["wind275_pair"],
          "note": "SCREENING_ERCOT.md's 7,921 was the contaminated column; "
                  "clean list = {5479, 5007}; 397x site-275 headline stands"},
      "groups": {"screening_pairs_v1": {
          k: {"readings": [round(v, 4) for v in readings[k].values()],
              "pairs": list(readings[k].keys()), "df": 2,
              "low_confidence": True} for k in OBJ}},
      "campaign_floor_T0": {k: max(readings[k].values()) for k in OBJ},
      "floor_rule": "campaign floor = max over groups; df<3 => LOW-CONFIDENCE, "
                    "verdicts need >=3x instead of >=2x; bimodal objectives "
                    "(total_cost, fixed_cost) never get Gaussian treatment",
      "bimodal": ["total_cost", "fixed_cost"],
      "v1_frozen": v1}
with open("noise_ruler_v2.json", "w") as f:
    json.dump(v2, f, indent=2); f.write("\n")
RULER = v2["campaign_floor_T0"]
print({k: round(v, 1) for k, v in RULER.items()})

decomposition verified: 7,921.1 = 5,007.0 clean + 2,914.1 h2
{'true_curt': 5479.1, 'h2': 2914.1, 'var_cost': 285620.6, 'reserve': 554.2, 'onoffs': 258.0, 'total_cost': 79451497.0, 'fixed_cost': 79504038.2}


/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_53419/1174093127.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = pd.read_csv(p); g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_53419/1174093127.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = pd.read_csv(p); g["is_pem"] = g["is_pem"].fillna(False).astype(bool)
/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_53419/1174093127.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is

## 2. T-M1 analogue — availability in cheap-LMP hours

Per candidate site (16 screening + nuclear gens 1 & 93): available energy in hours with
own-bus LMP < τ, τ ∈ {10, 15, 20, 30, 40} $/MWh (15 = RTS T-M1 threshold; 40 = scenario-C
dispatch threshold). Nuclear availability = PMax flat. This table is D6 *evidence*; the
design-space default remains the full lattice (Kay 09-19 no-feasibility-check precedent).

In [3]:
bus_csv = pd.read_csv(os.path.join(DATA, "bus.csv"))
id2name = dict(zip(bus_csv["Bus ID"].astype(int), bus_csv["Bus Name"].astype(str)))

lmps = pd.read_csv(os.path.join(ERCOT, "extracts", "base_2019", "bus_lmps.csv.gz"))
need_bus = sorted(set(eff.bus.astype(int)) | {107, 111})
name2id = {v: k for k, v in id2name.items()}
lmps = lmps[lmps.Bus.isin({id2name[b] for b in need_bus})].copy()
lmps["bus_id"] = lmps.Bus.map(name2id)
lmps["key"] = pd.to_datetime(lmps["Date"]).dt.dayofyear * 100 + lmps["Hour"]
LMP = {b: g.sort_values("key")["LMP"].to_numpy(float)
       for b, g in lmps.groupby("bus_id")}
nh = {b: len(v) for b, v in LMP.items()}
assert set(nh.values()) == {8760}, nh

def load_profile(fname):
    df = pd.read_csv(os.path.join(DATA, fname))
    df = df[df["Year"] == 2019].sort_values(["Month", "Day", "Period"])
    assert len(df) == 8760
    return df

wind = load_profile("REAL_TIME_wind.csv")
solar = load_profile("REAL_TIME_solar.csv")

CAND = [(r.site, int(r.bus), r.type, float(r.pmax)) for r in eff.itertuples()]
CAND += [("1", 107, "NUC", 2430.0), ("93", 111, "NUC", 2708.6)]
TAUS = [10, 15, 20, 30, 40]
rows = []
for site, bus, typ, pmax in CAND:
    if typ == "WIND":
        avail = wind[site].to_numpy(float)
    elif typ == "PV":
        avail = solar[site].to_numpy(float)
    else:
        avail = np.full(8760, pmax)
    lm = LMP[bus]
    tot = avail.sum()
    row = {"site": site, "bus": bus, "type": typ, "pmax_mw": pmax,
           "avail_total_mwh": tot}
    for t in TAUS:
        row[f"mwh_lt_{t}"] = float(avail[lm < t].sum())
        row[f"frac_lt_{t}"] = float(avail[lm < t].sum() / tot) if tot else np.nan
    rows.append(row)
TM1 = pd.DataFrame(rows)
TM1.to_csv("tm1_availability.csv", index=False)
with pd.option_context("display.width", 200):
    print(TM1[["site", "bus", "type", "avail_total_mwh"] +
              [f"frac_lt_{t}" for t in TAUS]].round(3).to_string(index=False))
print("\nNote: at tau=40 the fraction ~ 1 on this price distribution (mean LMP $14.53) — "
      "expected and stated; the spread across tau IS the exhibit.")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for site, bus, typ, pmax in CAND:
    m = TM1.site == site
    style = {"NUC": dict(ls="--", lw=2.2), "WIND": dict(ls="-"), "PV": dict(ls=":")}[typ]
    axes[0].plot(TAUS, TM1[m][[f"frac_lt_{t}" for t in TAUS]].iloc[0],
                 marker="o", ms=3, label=f"{site} ({typ}@{bus})", **style)
for b in sorted(set(b for _, b, _, _ in CAND)):
    x = np.sort(LMP[b]); axes[1].plot(x, np.arange(1, 8761) / 8760,
                                      lw=1.6 if b in (120, 107, 111) else 0.8,
                                      label=f"bus {b}" if b in (120, 107, 111, 2, 26) else None)
axes[1].set_xlim(-30, 60)
axes[0].set_xlabel("LMP threshold tau [$/MWh]"); axes[0].set_ylabel("fraction of available energy in LMP< tau hours")
axes[0].set_title("T-M1 analogue"); axes[0].legend(fontsize=5, ncol=2)
axes[1].set_xlabel("LMP [$/MWh]"); axes[1].set_ylabel("CDF"); axes[1].set_title("own-bus LMP CDFs")
axes[1].legend(fontsize=7)
fig.suptitle("Availability in cheap hours (tau=15 = RTS comparability column)")
fig.text(0.5, 0.005, FOOTER, ha="center", fontsize=7, style="italic")
fig.tight_layout(rect=[0, 0.02, 1, 1])
fig.savefig("figs/tm1_curves.png", dpi=150); plt.close(fig)
print("fig written")

site  bus type  avail_total_mwh  frac_lt_10  frac_lt_15  frac_lt_20  frac_lt_30  frac_lt_40
 275  120 WIND       4280224.48       0.930       0.967       0.985       0.995       0.996
 205   77 WIND       3647015.65       0.688       0.875       0.942       0.974       0.986
  20   32 WIND       3270585.70       0.668       0.877       0.919       0.968       0.986
 274  120 WIND       3067404.10       0.930       0.967       0.985       0.995       0.996
 204   28 WIND       2944284.57       0.693       0.884       0.948       0.976       0.988
  91   94 WIND       2750740.30       0.860       0.928       0.959       0.977       0.988
 206   22 WIND       2681475.01       0.776       0.899       0.948       0.970       0.984
  30   26   PV       2115800.50       0.931       0.979       0.986       0.992       0.997
 142   14   PV       1599445.65       0.781       0.919       0.950       0.975       0.987
 270  120 WIND       2375511.07       0.930       0.967       0.985       0.995 

fig written


## 3. Tier shares — RTS math-log §4.5 rule, positive-part

s_i = max(Δtrue_curt_i, 0) / Σ_j max(Δtrue_curt_j, 0) over the 16 distinct OAT designs
(Δ in the relief sign: positive = curtailment reduced). Independent dimension iff s_i ≥ 5%
AND own Δ ≥ 2× ruler (v1 true_curt ruler; LOW-CONFIDENCE ⇒ the ≥3× variant is also shown).
Negative responders (Δ < −2× ruler) listed separately, never pooled. Shared-bus clusters may
pool member shares (pv_324 precedent → bus-120 cluster).

In [4]:
ruler_tc = RULER["true_curt"]
eff["delta_relief"] = eff["relief"]
pos = eff.delta_relief.clip(lower=0.0)
eff["share"] = pos / pos.sum()
eff["x_ruler"] = eff.delta_relief / ruler_tc
eff["indep_2x"] = (eff.share >= 0.05) & (eff.delta_relief >= 2 * ruler_tc)
eff["indep_3x_lowconf"] = (eff.share >= 0.05) & (eff.delta_relief >= 3 * ruler_tc)
neg = eff[eff.delta_relief < -2 * ruler_tc]
tbl = eff.sort_values("share", ascending=False)[
    ["site", "bus", "type", "pmax", "delta_relief", "x_ruler", "share",
     "indep_2x", "indep_3x_lowconf"]]
tbl.to_csv("tier_shares.csv", index=False)
print(tbl.round(3).to_string(index=False))
b120 = eff[eff.bus == 120.0]
print(f"\nbus-120 cluster pooled share = {b120.share.sum():.3f} "
      f"(members {sorted(b120.site)})")
print(f"negative responders (< -2x ruler): "
      f"{[(r.site, round(r.x_ruler, 1)) for r in neg.itertuples()] or 'none'}")
print(f"\nindependent at >=5% & >=2x: {sorted(eff[eff.indep_2x].site)}")
print(f"independent at >=5% & >=3x (LOW-CONFIDENCE variant): {sorted(eff[eff.indep_3x_lowconf].site)}")

site   bus type     pmax  delta_relief  x_ruler  share  indep_2x  indep_3x_lowconf
 275 120.0 WIND 1039.000   2172937.626  396.588  0.229      True              True
 274 120.0 WIND  746.400   1568002.483  286.180  0.165      True              True
 270 120.0 WIND  579.000   1321264.974  241.147  0.139      True              True
 110   2.0 WIND  338.600    686606.340  125.314  0.072      True              True
  30  26.0   PV  926.667    674572.518  123.118  0.071      True              True
 146  75.0 WIND  268.000    548509.170  100.110  0.058      True              True
 140   2.0 WIND  250.600    528094.777   96.384  0.056      True              True
 163 120.0 WIND  185.400    425641.672   77.685  0.045     False             False
  59  26.0   PV  491.667    371528.272   67.808  0.039     False             False
 142  14.0   PV  703.333    347911.471   63.498  0.037     False             False
  91  94.0 WIND  785.500    281561.112   51.388  0.030     False             False
 206

## 4. Objective-pool first cut (n = 16; NON-USE clause: prioritizes T2 contrasts only)

Rank correlations among {true_curt(relief), var_cost, h2, starts} across the 16 distinct OAT
designs. Guards: raw AND gen_pmax-partialled Spearman side-by-side; |Δ| < 2× that objective's
ruler ⇒ tie (set to 0 before ranking); 5 designs share bus 120 (cluster structure beside every
coefficient). This measures cross-SITE agreement at fixed ω, not design-space redundancy.

In [5]:
P = pd.DataFrame({
    "relief": eff.delta_relief,
    "var_cost": eff.d_var_cost,
    "h2": eff.h2,
    "starts": eff.d_onoffs,
    "pmax": eff.pmax, "bus": eff.bus, "site": eff.site})
TIE_RULER = {"relief": RULER["true_curt"], "var_cost": RULER["var_cost"],
             "h2": RULER["h2"], "starts": RULER["onoffs"]}
Q = P.copy()
n_tied = {}
for c, r in TIE_RULER.items():
    m = Q[c].abs() < 2 * r
    n_tied[c] = int(m.sum())
    Q.loc[m, c] = 0.0
print("ties applied (|D| < 2x ruler):", n_tied)

objs = ["relief", "var_cost", "h2", "starts"]
def partial_spearman(x, y, z):
    rx, ry, rz = (stats.rankdata(v) for v in (x, y, z))
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    return float(np.corrcoef(ex, ey)[0, 1])

rows = []
for i, a in enumerate(objs):
    for b in objs[i + 1:]:
        rows.append({"pair": f"{a}|{b}",
                     "spearman_raw": round(float(stats.spearmanr(Q[a], Q[b]).statistic), 3),
                     "spearman_partial_pmax": round(partial_spearman(Q[a], Q[b], Q.pmax), 3),
                     "n": 16, "n_bus120": 5})
CORR = pd.DataFrame(rows)
CORR.to_csv("pool_correlations.csv", index=False)
print(CORR.to_string(index=False))
print("\nNON-USE CLAUSE: these correlations may not add/drop pool members — "
      "they only prioritize T2 contrasts.")

ties applied (|D| < 2x ruler): {'relief': 0, 'var_cost': 0, 'h2': 0, 'starts': 10}
           pair  spearman_raw  spearman_partial_pmax  n  n_bus120
relief|var_cost        -0.368                 -0.364 16         5
      relief|h2        -0.041                  0.240 16         5
  relief|starts         0.566                  0.685 16         5
    var_cost|h2         0.782                  0.411 16         5
var_cost|starts        -0.814                 -0.576 16         5
      h2|starts        -0.755                 -0.264 16         5

NON-USE CLAUSE: these correlations may not add/drop pool members — they only prioritize T2 contrasts.


## 5. Congestion case study — bus 120 (the NXT-D4 exhibit)

In [6]:
branch = pd.read_csv(os.path.join(DATA, "branch.csv"))
b120_lines = branch[(branch["From Bus"] == 120) | (branch["To Bus"] == 120)]
ls = pd.read_csv(os.path.join(ERCOT, "extracts", "base_2019", "line_summary.csv"))
ls["line"] = ls["Line"].astype(str)
b120_ls = ls[ls.line.isin(b120_lines["UID"].astype(str))]
print("lines touching bus 120 (base year):")
print(b120_ls[["line", "cont_rating", "max_abs_flow", "p99_abs_flow",
               "hours_ge_99pct"]].to_string(index=False))

lm120, lmref = LMP[120], np.median(np.vstack([LMP[b] for b in LMP]), axis=0)
share0_120 = float((lm120 <= 1.0).mean()); share0_ref = float((lmref <= 1.0).mean())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(np.sort(lm120), np.arange(1, 8761) / 8760, lw=2, label="bus 120")
axes[0].plot(np.sort(lmref), np.arange(1, 8761) / 8760, lw=1.2, ls="--",
             label="median candidate bus")
axes[0].set_xlim(-30, 60); axes[0].set_xlabel("LMP [$/MWh]"); axes[0].set_ylabel("CDF")
axes[0].set_title(f"bus-120 LMP: {share0_120:.0%} of hours <= $1 (ref {share0_ref:.0%})")
axes[0].legend(fontsize=8)
colors = ["crimson" if b == 120 else "tab:blue" for b in eff.bus]
axes[1].bar(range(16), eff.relief_per_h2, color=colors)
axes[1].set_xticks(range(16), eff.site, fontsize=7, rotation=45)
axes[1].set_ylabel("system relief per MWh H2")
axes[1].set_title("relief/H2 (red = bus-120 pocket)")
fig.suptitle("Congestion pocket case study (NXT-D4 exhibit)")
fig.text(0.5, 0.005, FOOTER, ha="center", fontsize=7, style="italic")
fig.tight_layout(rect=[0, 0.02, 1, 1])
fig.savefig("figs/bus120_case_study.png", dpi=150); plt.close(fig)
print("fig written")

lines touching bus 120 (base year):
line  cont_rating  max_abs_flow  p99_abs_flow  hours_ge_99pct
L100   1493.89381    511.977147    275.533618               0
L113   1207.06625   1029.958442    876.639311               0
L197   1326.57776    515.570764    305.398576               0
L198   1493.89381   1100.229227    651.721283               0
L240   1326.57776    540.794562    347.784644               0
L252    700.00000   2070.875900   1226.684282            6769
  L4   1326.57776    877.062748    565.177423               0
  L5   1326.57776    397.442690    256.111248               0


fig written


## 6. Fuel-accounting note inputs (g1-analogue; propose-first, pre-registered BEFORE T1)

In [7]:
gb = gens(os.path.join(ERCOT, "extracts", "base_2019", "gen_summary.csv")).set_index("Generator")
g1 = gb.loc["1"]
note = {
    "patch_mechanism": "multi_pem/parameters.py:27-43 on an egret-0.6.2 ERCOT thermal dict: "
                       "parser emits p_fuel (fuel_curve) and NO p_cost (verified locally "
                       "2026-10-08) -> the patch takes the del-p_fuel branch and installs the "
                       "flat band p_cost [[p_min,0],[p_max, cap*B]] + forced commitment",
    "gen1_base_booked_unit_cost_usd": float(g1["Unit Cost"]),
    "gen1_base_output_mwh": float(g1["output_mwh"]),
    "gen1_base_starts": float(g1["starts"]),
    "gen93_base": {"output_mwh": float(gb.loc["93", "output_mwh"]),
                   "unit_cost": float(gb.loc["93", "Unit Cost"]),
                   "starts": float(gb.loc["93", "starts"]),
                   "fact": "gen 93 NEVER COMMITS in the base year"},
    "parser_facts": {"fuel_cost_usd_per_mmbtu": 0.81035,
                     "p_fuel_at_pmin_mmbtu_h": 2623266.49,
                     "marginal_mmbtu_per_mwh": 15.74,
                     "units_note": "p_fuel curve magnitude does not reconcile with the "
                                   "booked $3.18B/yr at face value — normalization is a "
                                   "recorded open question (PI-meeting material, not "
                                   "blocking); the OBSERVED base Unit Cost is authoritative "
                                   "for the deletion magnitude"},
    "rule": "T1 nuclear var_cost readings reported raw AND fuel-cost-adjusted, each with the "
            "synthetic-cost caveat; adjudication = PI (RTS g1/NXT-D7 never imported silently)",
}
with open("fuel_note.json", "w") as f:
    json.dump(note, f, indent=2); f.write("\n")
print(json.dumps(note, indent=2)[:900])

{
  "patch_mechanism": "multi_pem/parameters.py:27-43 on an egret-0.6.2 ERCOT thermal dict: parser emits p_fuel (fuel_curve) and NO p_cost (verified locally 2026-10-08) -> the patch takes the del-p_fuel branch and installs the flat band p_cost [[p_min,0],[p_max, cap*B]] + forced commitment",
  "gen1_base_booked_unit_cost_usd": 3184913484.851423,
  "gen1_base_output_mwh": 3304484.546403,
  "gen1_base_starts": 25.0,
  "gen93_base": {
    "output_mwh": 0.0,
    "unit_cost": 0.0,
    "starts": 0.0,
    "fact": "gen 93 NEVER COMMITS in the base year"
  },
  "parser_facts": {
    "fuel_cost_usd_per_mmbtu": 0.81035,
    "p_fuel_at_pmin_mmbtu_h": 2623266.49,
    "marginal_mmbtu_per_mwh": 15.74,
    "units_note": "p_fuel curve magnitude does not reconcile with the booked $3.18B/yr at face value \u2014 normalization is a recorded open question (PI-meeting material, not blocking); the OBSERVED base


/var/folders/p0/8_mj7nxn1td_x1r_61m5pqh40000gn/T/ipykernel_53419/1174093127.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  g = pd.read_csv(p); g["is_pem"] = g["is_pem"].fillna(False).astype(bool)


In [8]:
summary = {
    "ruler_v2": v2["campaign_floor_T0"],
    "tm1_sites": len(TM1),
    "tier_shares_top": tbl.head(8)[["site", "share", "x_ruler"]].to_dict("records"),
    "bus120_pooled_share": float(b120.share.sum()),
    "negative_responders": [(r.site, round(r.x_ruler, 2)) for r in neg.itertuples()],
    "pool_correlations": CORR.to_dict("records"),
    "gen93_never_commits": True,
}
with open("t0_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=float)
print("t0_summary.json written; outputs:",
      sorted(f for f in os.listdir(".") if f.endswith((".csv", ".json"))))

t0_summary.json written; outputs: ['fuel_note.json', 'noise_ruler_v2.json', 'pool_correlations.csv', 't0_summary.json', 'tier_shares.csv', 'tm1_availability.csv']
